In [ ]:
job_id = str(uuid.uuid4())
job_name = 'DM_DIM_CAS_CASES_LOAD'
layer_name = 'DATAMART'
start_time = datetime.now()
rows_loaded = 0
rows_failed = 0
run_status = 'SUCCESS'
error_message = None

try:
    # Step 1: Create temp table from source
    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_DM_CAS_CASES AS
    SELECT DISTINCT
        CAS_ID,
        CASE_NAME,
        CASE_TYPE_AV_ID,
        CASE_TYPE_DESC,
        UNIT_ORGN_NAME,
        AREA_ORGN_NAME,
        REGION_ORGN_NAME,
        CURRENT_CASE_STATUS_CODE_AV_ID,
        CURRENT_CASE_STATUS_CODE_DESC,
        CURRENT_CASE_STATUS_DATE,
        CASE_WORKER_NAME,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(CAS_ID), ''),
            COALESCE(CASE_NAME, ''),
            COALESCE(CASE_TYPE_AV_ID, ''),
            COALESCE(CASE_TYPE_DESC, ''),
            COALESCE(TO_VARCHAR(UNIT_ORGN_NAME), ''),
            COALESCE(TO_VARCHAR(AREA_ORGN_NAME), ''),
            COALESCE(TO_VARCHAR(REGION_ORGN_NAME), ''),
            COALESCE(CURRENT_CASE_STATUS_CODE_AV_ID, ''),
            COALESCE(CURRENT_CASE_STATUS_CODE_DESC, ''),
            COALESCE(TO_VARCHAR(CURRENT_CASE_STATUS_DATE), ''),
            COALESCE(TO_VARCHAR(CASE_WORKER_NAME), '')
        ), 256) AS CHECKSUM
    FROM {DB}.{DATAMART}.{DM_SOURCE}
    WHERE CAS_ID IS NOT NULL
    """).collect()

    # Step 2: SCD2 - Expire old records where checksum changed
    session.sql(f"""
    UPDATE {DB}.{DATAMART}.DIM_DM_CAS_CASES_DT tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_DM_CAS_CASES src
    WHERE tgt.CAS_ID = src.CAS_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    # Step 3: SCD2 - Insert new/changed versions
    insert_result = session.sql(f"""
    INSERT INTO {DB}.{DATAMART}.DIM_DM_CAS_CASES_DT (
        CAS_ID, CASE_NAME, CASE_TYPE_AV_ID, CASE_TYPE_DESC,
        UNIT_ORGN_NAME, AREA_ORGN_NAME, REGION_ORGN_NAME,
        CURRENT_CASE_STATUS_CODE_AV_ID, CURRENT_CASE_STATUS_CODE_DESC, CURRENT_CASE_STATUS_DATE,
        CASE_WORKER_NAME, CREATED_DATE, UPDATED_DATE, CHECKSUM
    )
    SELECT
        src.CAS_ID, src.CASE_NAME, src.CASE_TYPE_AV_ID, src.CASE_TYPE_DESC,
        src.UNIT_ORGN_NAME, src.AREA_ORGN_NAME, src.REGION_ORGN_NAME,
        src.CURRENT_CASE_STATUS_CODE_AV_ID, src.CURRENT_CASE_STATUS_CODE_DESC, src.CURRENT_CASE_STATUS_DATE,
        src.CASE_WORKER_NAME, CURRENT_TIMESTAMP(), NULL, src.CHECKSUM
    FROM TEMP_DM_CAS_CASES src
    WHERE NOT EXISTS (
        SELECT 1 FROM {DB}.{DATAMART}.DIM_DM_CAS_CASES_DT tgt
        WHERE tgt.CAS_ID = src.CAS_ID
          AND tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), rows_loaded, rows_failed, run_status, None)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_CAS_CASES loaded. Rows: {rows_loaded}')
    print(f'SUCCESS | Rows Loaded: {rows_loaded}')

except Exception as error:
    run_status = 'FAILED'
    rows_failed = 1
    error_message = str(error)
    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), 0, rows_failed, run_status, error_message)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_CAS_CASES failed: {error_message}')
    print(f'FAILED: {error_message}')